In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = "4,5,6,7"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from torch.utils.data import DataLoader

/mnt/petrelfs/zhangshilin/anaconda3/envs/deepscaler/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-04 11:31:47,888	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
model_path = "/mnt/petrelfs/share_data/huzican/Qwen2.5-7B-orz-tok"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path).cuda()

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.53it/s]


In [3]:
train_data_path = "dataset/valid.parquet"
train_dataset = RLHFDataset(parquet_files=train_data_path,
                            tokenizer=tokenizer,
                            prompt_key='prompt',
                            max_prompt_length=1000,
                            filter_prompts=True,
                            return_raw_chat=False,
                            truncation='error')

train_dataloader = DataLoader(dataset=train_dataset,
                            batch_size=3,
                            shuffle=True,
                            drop_last=True,
                            collate_fn=collate_fn)

original dataset len: 2019
filter dataset len: 2017


In [4]:
for test_data in train_dataloader:
    print(test_data.keys())
    seq = tokenizer.batch_decode(test_data['input_ids'],skip_special_tokens=True)
    # print(seq)
    # 准备输入
    input_ids = test_data['input_ids'].to(model.device)
    print(input_ids.shape)
    attention_mask = test_data['attention_mask'].to(model.device)
    group_rollout = []


    output = model(input_ids=input_ids,
                   attention_mask=attention_mask,
                   output_hidden_states=True)
    print(f'len: {len(output.hidden_states)}')
    print(f'shape: {output.hidden_states[-1].shape}')
    # print(f'last hidden state: {output.last_hidden_states.shape}')

    torch.cuda.empty_cache()
    break
                   

dict_keys(['input_ids', 'attention_mask', 'position_ids', 'data_source', 'ability', 'reward_model', 'extra_info', 'index'])
torch.Size([3, 1000])
len: 29
shape: torch.Size([3, 1000, 3584])
